# SCADA Functionality Test Visualiser

**Author:** Erick Chauke

SCADA stands for Supervisory Control and Data Acquisition, the system that controls a power plant and records its measurements second by second. Every grid-code functionality test leaves behind one of these logger spreadsheets, and right now each one is checked by eye against the acceptance procedure. This notebook is the start of doing that checking automatically. You paste a test export into the `data/` folder, run the notebook, and it parses the channels, lines the measured values up against their setpoints and control modes, and draws the plots that show whether the plant behaved the way each part of the acceptance procedure requires. The procedure is the SCADA Functionality Test Record (Rev 3) issued by NCSS, the National Control System Support group of the national grid operator [1].

The point is that it is general. Nothing here is wired to one plant or one capture. The single config cell below is the only thing that changes from one test to the next, so the same notebook handles any site whose logger follows the standard layout. New test sections are added one at a time as the tool grows, so over time it becomes a single place to visualise and sign off the whole procedure. The first section built is the curtailment, or absolute production constraint, test, which checks that the plant can hold its output below a commanded ceiling on demand.

## Setup

The cell below is the only place anything is set. To run against a new test, drop that spreadsheet into `data/` and run all cells, and nothing else is touched. It locates the workbook, records the site and time zone, sets the event windows worth highlighting, and creates `outputs/` if it is not already there. Figures are saved under `outputs/` with the source workbook name as a prefix, so results from different plants never overwrite one another.

In [2]:
# Single config cell. To point this notebook at a different test, drop the new
# spreadsheet into the data/ folder and (only if more than one file is present)
# adjust INPUT_GLOB below. Nothing else in the notebook is edited to swap sites.

from pathlib import Path

DATA_DIR = Path("data")          # input files live here (gitignored)
INPUT_GLOB = "*.xlsx"            # pattern that selects the OPC logger workbook
OUTPUT_DIR = Path("outputs")     # every figure is saved here

SITE_NAME = "Hartebeesthoek"     # plant name shown in titles
TIME_ZONE_LABEL = "UTC"         # the logger timestamps are in this zone

# Highlighted event windows, used to slice and annotate the plots.
# Each entry maps a label to a (start, end) pair of timestamps.
EVENT_WINDOWS = {
    "curtailment_35MW": ("2026-05-27 08:14:30", "2026-05-27 08:17:00"),
}

# Resolve the single input workbook without hard-coding its (confidential) name.
_candidates = sorted(DATA_DIR.glob(INPUT_GLOB))
assert len(_candidates) >= 1, f"no file matching {INPUT_GLOB} found in {DATA_DIR}"
INPUT_FILE = _candidates[0]
STEM = INPUT_FILE.stem           # used to namespace output figures per site

OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Site: {SITE_NAME} | timezone: {TIME_ZONE_LABEL}")
print(f"Input workbook resolved from {DATA_DIR}/ ({len(_candidates)} match) | stem captured")
print(f"Highlighted events: {list(EVENT_WINDOWS)}")
print(f"Figures will be written to {OUTPUT_DIR}/")

Site: Hartebeesthoek | timezone: UTC
Input workbook resolved from data/ (1 match) | stem captured
Highlighted events: ['curtailment_35MW']
Figures will be written to outputs/


## Data ingestion and inspection

Before trusting any figure, this section loads the workbook and reports what is actually inside it: how many sheets there are, the row and column counts, the channel names, and the type of each column. The output here is a structural check rather than a plot, and it is the moment to confirm the data matches what the test was expected to capture.

In [ ]:
import pandas as pd

# Load every sheet so the real structure is confirmed before any narrative is built
# on it. sheet_name=None keeps a multi-sheet workbook from being silently reduced to
# its first tab, and na_values catches the string sentinels that Excel exports leave
# behind so they read as missing rather than as text.
sheets = pd.read_excel(INPUT_FILE, sheet_name=None, na_values=["NULL", "None"])

print(f"Sheets found: {len(sheets)}")
for name, frame in sheets.items():
    print(f"  '{name}': {frame.shape[0]} rows x {frame.shape[1]} columns")

# This logger export is a single sheet. Take it as the working frame.
raw = next(iter(sheets.values()))

print("\nColumns and dtypes:")
for col, dtype in raw.dtypes.items():
    print(f"  {col:<22} {dtype}")

print("\nFirst rows:")
raw.head()

## Data cleaning and parsing

The raw sheet is not ready to plot yet, and three things need fixing first. The time of day is stored as a text string wrapped in quotation marks, such as `'08:00:47'`, so the quotes are stripped and the value is joined to the calendar date to form a single timestamp that becomes the index of every later plot. The control-mode columns are logged as 0 and 1 integers, which are turned into false and true so they read as off and on. Finally every measurement channel is forced to a numeric type, so that any stray text left behind by the export becomes a missing value instead of quietly breaking a calculation.

Two abbreviations appear from here on. A setpoint, written SP, is the value the control centre commands the plant to follow. The POC, or Point of Connection, is the point where the plant meets the grid, and it is where these measurements are taken.

In [4]:
df = raw.copy()

# The time of day arrives as a quote-wrapped string like "'08:00:47'". Strip the quotes
# and any whitespace, then read it as a time offset within the day.
time_str = df["Time UTC(NC2)"].astype(str).str.strip().str.strip("'\"")
time_part = pd.to_timedelta(time_str)

# The Date column already parsed as a datetime at midnight. Combine its calendar date
# with the time of day to get one full timestamp, and make it the index.
date_part = pd.to_datetime(df["Date"]).dt.normalize()
df.insert(0, "timestamp", date_part + time_part)
df = df.set_index("timestamp").drop(columns=["Date", "Time UTC(NC2)"])

# The six control-mode flags are logged as 0 and 1 integers. Read them as on and off.
mode_cols = [c for c in df.columns if c.startswith("Mode")]
df[mode_cols] = df[mode_cols].astype(bool)

# Every remaining channel is a measurement or a setpoint. Force it to a numeric type so
# any stray text left by the export becomes a missing value instead of breaking a sum.
measure_cols = [c for c in df.columns if c not in mode_cols]
df[measure_cols] = df[measure_cols].apply(pd.to_numeric, errors="coerce")

step = df.index.to_series().diff().median()
print(f"Parsed {len(df)} rows spanning {df.index.min()} to {df.index.max()} {TIME_ZONE_LABEL}")
print(f"Median sample step: {step}")
print(f"Mode flags cast to boolean: {mode_cols}")
print(f"Missing values after numeric coercion: {int(df[measure_cols].isna().sum().sum())}")
df.head()

Parsed 7147 rows spanning 2026-05-27 08:00:47 to 2026-05-27 09:59:53 UTC
Median sample step: 0 days 00:00:01
Mode flags cast to boolean: ['Mode:Q', 'Mode:V', 'Mode:PF', 'Mode: Active Power', 'Mode: p-Delta', 'Mode: Power Gradient']
Missing values after numeric coercion: 0


,POC: P (MW),POC: Q (MVAr),POC: Freq (Hz),POC: PF,POC: Average Voltage (KV),SP: P (MW),SP: Q (MVAr),SP:Voltage (kV),SP: PF,SP: P-Delta (%),...,SP:Ramp up (MW/min),SP:Ramp down (MW/min),Mode:Q,Mode:V,Mode:PF,Mode: Active Power,Mode: p-Delta,Mode: Power Gradient,SP:Droop V (%),SP:Droop F (%)
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-05-27 08:00:47,43.776010,-3.911456,49.8469,-0.996032,137.41379,140,0,137,1,0,...,5,10,False,True,False,False,False,False,4,4
2026-05-27 08:00:48,43.781395,-3.988405,49.8480,-0.995876,137.45483,140,0,137,1,0,...,5,10,False,True,False,False,False,False,4,4
2026-05-27 08:00:49,43.686543,-3.873350,49.8506,-0.996093,137.50957,140,0,137,1,0,...,5,10,False,True,False,False,False,False,4,4
2026-05-27 08:00:50,43.683760,-3.826938,49.8555,-0.996185,137.46852,140,0,137,1,0,...,5,10,False,True,False,False,False,False,4,4
2026-05-27 08:00:51,43.658350,-3.850763,49.8547,-0.996133,137.43090,140,0,137,1,0,...,5,10,False,True,False,False,False,False,4,4
